# 02 — Distance & the scale trap

*Module 01 · k-Nearest Neighbours — notebook 2 of 6*

In notebook 1 you built k-NN and saw that everything hinges on one word: **nearest**. A prediction is
only as trustworthy as the distances behind it. So this notebook looks hard at distance — and
uncovers a trap that catches every distance-based method.

Here is the catch: k-NN has no opinion about your features' units. If one feature happens to be
measured on a much larger scale than another, it quietly takes over the distance, and the neighbours
— and your predictions — follow it instead of the data's real shape. The fix is one you already met
in module 00: **standardization**.

**Prerequisites:** notebook 01 (the neighbourhood vote), and from module 00: notebook 02 (Euclidean
distance) and notebook 11 (standardization, fit on the training set).

**What you will be able to do**

- Compute Euclidean (L2) and Manhattan (L1) distance by hand, and see how the choice can change who
  counts as "nearest".
- Explain why k-NN, being pure distance, is at the mercy of feature scale.
- Watch accuracy collapse when one feature is rescaled — and recover it by standardizing.
- Judge honestly how much the metric matters compared to the scale.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

from ml_course import viz
from ml_course.colors import CLASS_CYCLE, COLORS

# Fix the seed so every run tells the same story.
np.random.seed(0)
viz.use_course_style()

X, y = make_moons(n_samples=300, noise=0.30, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=0
)


# The by-hand k-NN vote from notebook 1, now with a switch between two distances.
# metric: "euclidean" (L2, straight line) or "manhattan" (L1, city-block).
def knn_predict(X_ref, y_ref, queries, k=5, metric="euclidean"):
    X_ref = np.asarray(X_ref, dtype=float)
    queries = np.atleast_2d(np.asarray(queries, dtype=float))
    predictions = []
    for q in queries:
        if metric == "euclidean":
            dist = np.sqrt(((X_ref - q) ** 2).sum(axis=1))
        else:  # manhattan (L1)
            dist = np.abs(X_ref - q).sum(axis=1)
        nearest = np.argsort(dist)[:k]
        predictions.append(np.bincount(y_ref[nearest], minlength=2).argmax())
    return np.array(predictions)


print(f"training points: {X_train.shape[0]}   test points: {X_test.shape[0]}")

## A recap, and the question this notebook asks

Three ideas carry this notebook:

- **The vote** (notebook 01) — k-NN labels a point by the majority vote of its k nearest neighbours.
  "Nearest" is the whole game.
- **Euclidean distance** (notebook 02) — the straight-line distance between two points: the square
  root of the summed squared differences across features.
- **Standardization** (notebook 11) — rescaling each feature to mean 0 and standard deviation 1, with
  the mean and standard deviation computed on the **training set only**.

The new question is easy to state and easy to underestimate: *what are the features measured in?*
Because k-NN is nothing but distance, the answer turns out to decide everything. That is the **scale
trap**.

## Two ways to measure "near"

"Distance" is not a single thing. Two common choices:

- **Euclidean distance (L2)** — the straight line, as the crow flies: the square root of the summed
  squared differences. This is the one from notebook 02, and k-NN's usual default.
- **Manhattan distance (L1)** — the city-block walk: the sum of the absolute differences, one axis at
  a time, as if you could only move along streets.

For two points $p$ and $q$ in two dimensions:

- Euclidean: $\sqrt{(p_1 - q_1)^2 + (p_2 - q_2)^2}$
- Manhattan: $|p_1 - q_1| + |p_2 - q_2|$

They usually disagree on the exact numbers — and sometimes even on *who is nearest*.

In [ ]:
# From a query q, compare two candidate neighbours A and B under each metric.
q = np.array([0.0, 0.0])
A = np.array([1.0, 0.0])
B = np.array([0.7, 0.7])

for name, p in (("A", A), ("B", B)):
    euclidean = np.sqrt(((p - q) ** 2).sum())
    manhattan = np.abs(p - q).sum()
    print(f"{name} = {p.tolist()}:  Euclidean {euclidean:.3f}   Manhattan {manhattan:.3f}")

**Read the output.** Look at which point each metric calls *nearest* to `q`. By Euclidean distance,
**B** is closer (0.990 < 1.000) — the diagonal shortcut is short as the crow flies. By Manhattan
distance, **A** is closer (1.000 < 1.400) — walking along streets, the diagonal point costs you both a
horizontal and a vertical block. Same two points, same query, opposite answers. So the metric is a
real choice. But hold that thought: we are about to meet something that affects k-NN far more.

## k-NN is pure distance, so it is at the mercy of scale

Euclidean distance sums the squared differences across features. If one feature ranges over tens or
hundreds while another ranges over a fraction of one, the large-range feature dominates that sum
almost completely — the small one barely registers. Since k-NN ranks neighbours purely by this
distance, it ends up listening to one feature and ignoring the other, whatever the data's real shape.

Let us provoke this on purpose: take our moons and stretch one feature's scale by a factor of 50, as
if someone had recorded it in different units. The data's shape is unchanged — only the yardstick on
one axis.

In [ ]:
# Stretch feature 2 by 50 - same data, a different unit on one axis.
SCALE = 50
X_scaled = X.copy()
X_scaled[:, 1] = X_scaled[:, 1] * SCALE

print(f"feature 1 span:          {np.ptp(X[:, 0]):.2f}")
print(f"feature 2 span (x{SCALE}):     {np.ptp(X_scaled[:, 1]):.1f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, data, title in (
    (axes[0], X, "original"),
    (axes[1], X_scaled, f"feature 2 rescaled x{SCALE}"),
):
    for cls in (0, 1):
        mask = y == cls
        ax.scatter(
            data[mask, 0],
            data[mask, 1],
            color=CLASS_CYCLE[cls],
            edgecolor=COLORS["text"],
            linewidth=0.4,
            s=30,
            alpha=0.8,
            label=f"class {cls}",
        )
    ax.set_xlabel("feature 1")
    ax.set_ylabel("feature 2")
    ax.set_title(title)
    ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

**Read the figure.** At a glance the two scatters look the same — because each panel autoscales its
own axes to fit the data. The trap hides in the axis numbers: on the right, feature 2 now runs out
past ±80, while feature 1 still sits within about ±3. Our eyes are reassured by the plot's rescaling;
k-NN's distance is not, because it works on the raw magnitudes. Feature 2's span is now about 34 times
feature 1's (149.9 versus 4.40) — and because Euclidean distance adds up *squared* differences,
feature 2 weighs on the order of 34², roughly a thousand times, more in that sum. It has all but
erased feature 1, even though we changed nothing about the classes.

In [ ]:
# Same split recipe as notebook 1, now on the rescaled data.
Xs_train, Xs_test, ys_train, ys_test = train_test_split(
    X_scaled, y, test_size=0.30, stratify=y, random_state=0
)

acc_original = (knn_predict(X_train, y_train, X_test) == y_test).mean()
acc_scaled = (knn_predict(Xs_train, ys_train, Xs_test) == ys_test).mean()

print(f"k-NN(5) test accuracy on original features:   {acc_original:.3f}")
print(f"k-NN(5) test accuracy after rescaling x{SCALE}:    {acc_scaled:.3f}")

**Read the output.** Accuracy dropped from 0.956 to 0.733 — a large fall — and we changed nothing
about the data except the unit on one axis. k-NN did exactly what we asked: it ranked neighbours by a
distance now dominated by feature 2, so two points that are close in the full picture can look far
apart to it, and it mislabels them. Nothing is wrong with k-NN; the yardstick is wrong.

## The fix: put every feature on the same footing

The cure is standardization, from notebook 11: for each feature, subtract its mean and divide by its
standard deviation, so every feature ends up centred at 0 with spread 1. No feature can then dominate
the distance by virtue of its units alone.

One rule travels with it, and it is not optional: compute the mean and standard deviation on the
**training set only**, then apply those same numbers to the test set. The test set must stay sealed —
letting it influence the scaling would leak information and flatter the score (notebook 11).

In [ ]:
# Standardize using TRAINING statistics only (notebook 11), then apply to both sets.
mu = Xs_train.mean(axis=0)
sigma = Xs_train.std(axis=0)
Xs_train_std = (Xs_train - mu) / sigma
Xs_test_std = (Xs_test - mu) / sigma

acc_standardized = (knn_predict(Xs_train_std, ys_train, Xs_test_std) == ys_test).mean()

print(f"rescaled, no standardization: {acc_scaled:.3f}")
print(f"rescaled, then standardized:  {acc_standardized:.3f}")

**Read the output.** Standardizing brought accuracy back from 0.733 to 0.956 — exactly the original
score. That *exact* recovery is special to this demonstration: because we created the problem with a
pure rescaling, undoing the scale undoes the damage completely. On real data, standardizing usually
helps a distance-based model, though it need not return an identical number. The point that always
holds: distance-based methods deserve features on a comparable scale.

## Seeing it: the decision boundary

To picture what rescaling does to k-NN, we will draw its **decision boundary** — the predicted class
at every point of the plane — for the rescaled data and the standardized data side by side. The
course's boundary plotter expects an object with `fit` and `predict` methods, so we wrap our by-hand
vote (the very same `knn_predict`) in a tiny `ByHandKNN` object of that shape. No new algorithm — only
a convenient wrapper. (That `fit`/`predict` shape is exactly the estimator interface we will meet for
real in notebook 4.)

In [ ]:
# A tiny by-hand k-NN object, so we can reuse the course's boundary plotter.
class ByHandKNN:
    def __init__(self, k=5, metric="euclidean"):
        self.k = k
        self.metric = metric

    def fit(self, X_ref, y_ref):
        self.X_ref = np.asarray(X_ref, dtype=float)
        self.y_ref = np.asarray(y_ref)
        return self

    def predict(self, queries):
        return knn_predict(self.X_ref, self.y_ref, queries, self.k, self.metric)


fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
viz.plot_decision_boundary(
    ByHandKNN().fit(Xs_train, ys_train), Xs_train, ys_train, resolution=200, ax=axes[0]
)
viz.plot_decision_boundary(
    ByHandKNN().fit(Xs_train_std, ys_train), Xs_train_std, ys_train, resolution=200, ax=axes[1]
)
axes[0].set_title(f"rescaled x{SCALE} - raw (broken)")
axes[1].set_title("standardized (recovered)")
for ax in axes:
    ax.set_xlabel("feature 1")
    ax.set_ylabel("feature 2")
fig.tight_layout()
plt.show()

**Read the figure.** On the left, with feature 2 rescaled and no standardization, the decision
regions are nearly horizontal bands: because feature 2 dominates the distance, k-NN groups points by
their vertical position alone and slices straight through the crescents. On the right, after
standardizing, the familiar two-moon boundary returns. We did not change k-NN or the data — we
changed the yardstick, and "near" went back to meaning what we intend.

In [ ]:
# With the scale fixed, does the choice of metric matter much here?
pred_euclidean = knn_predict(Xs_train_std, ys_train, Xs_test_std, metric="euclidean")
pred_manhattan = knn_predict(Xs_train_std, ys_train, Xs_test_std, metric="manhattan")

print(f"standardized, Euclidean (L2): {(pred_euclidean == ys_test).mean():.3f}")
print(f"standardized, Manhattan (L1): {(pred_manhattan == ys_test).mean():.3f}")

**Read the output.** Switching the metric barely moves accuracy here — 0.956 versus 0.944 is a single
test point out of 90, well within the noise of one split. Fixing the *scale*, by contrast, moved
accuracy by more than twenty points (0.733 to 0.956). That gap in magnitude is the lesson: on data
like this, **getting the scale right matters far more than choosing the metric.** The metric is still a
genuine dial — we tune it as a hyperparameter in notebook 4, and study the geometry of different
distances in notebook 6 — but it is a fine adjustment next to the scale.

## Your turn

Reason these through before reaching for code.

1. You have two features: one in kilometres (values around 0–10), one in millimetres (values around
   0–5000). Before any standardization, which feature will dominate a Euclidean distance, and what
   will standardizing do to that imbalance?
2. Standardization here used the training mean and standard deviation, then applied them to the test
   set. Why must the statistics come from the training set only? What would go wrong if you
   standardized the whole dataset before splitting? (Notebook 11.)
3. Suppose you standardize a dataset and the accuracy does not change at all. What does that tell you
   about the original features' scales?

## What you built

You looked hard at the one word k-NN depends on — *nearest* — and found the trap underneath it. You
computed Euclidean and Manhattan distance by hand and saw the metric can change who counts as nearest.
Then you watched a single rescaled feature drag k-NN's accuracy from 0.956 down to 0.733, and you
brought it back with standardization, fit on the training set only. And you saw the honest ranking:
scale first, metric second.

**Vocabulary**

- **Euclidean distance (L2)** — straight-line distance: the root of the summed squared differences.
- **Manhattan distance (L1)** — city-block distance: the sum of the absolute differences.
- **feature scale** — the range of values a feature takes, set by its units.
- **the scale trap** — a large-range feature dominating the distance, so k-NN follows it instead of
  the data's shape.
- **standardization (z-score)** — centring each feature to mean 0, spread 1, using training statistics
  only.
- **scale-invariance** — the property a model gains once standardized: its predictions no longer
  depend on the features' units.

## References

- G. James, D. Witten, T. Hastie, R. Tibshirani (2021). *An Introduction to Statistical Learning*,
  §2.2.3 (k-NN) and ch. 4 (the classification lab, which standardizes features before applying k-NN —
  for the reason this notebook demonstrates). DOI:
  [10.1007/978-1-0716-1418-1](https://doi.org/10.1007/978-1-0716-1418-1).
- T. Hastie, R. Tibshirani, J. Friedman (2009). *The Elements of Statistical Learning*, §2.5 & §13.3
  (k-NN; the role of distance). DOI:
  [10.1007/978-0-387-84858-7](https://doi.org/10.1007/978-0-387-84858-7).

---

*Previous: 01 — Predict by the neighbourhood vote* · *Next: 03 — The k dial*